Start Time: 3:50 PM PKT
End Time: 6:38 PM PKT
Total Duration: 3 Hours

# Week 2 Challenge: Beat the Baseline

## Objective
Predict whether a hotel booking will be cancelled (`is_canceled`).

## Evaluation
ROC AUC using predicted probabilities.

## Baseline Score
0.710

## Time Limit
3 hours

In [28]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load dataset
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print(df.head())

Dataset loaded successfully!
Shape: (119390, 32)
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                        0          

In [29]:
# Separate features and target
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

# Required train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training features: (95512, 31)
Testing features: (23878, 31)

Training target distribution:
is_canceled
0    0.629586
1    0.370414
Name: proportion, dtype: float64

Testing target distribution:
is_canceled
0    0.629575
1    0.370425
Name: proportion, dtype: float64


In [30]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Baseline features
baseline_features = [
    "lead_time",
    "adr",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "previous_cancellations"
]

# Select features
X_train_base = X_train[baseline_features]
X_test_base = X_test[baseline_features]

# Create baseline pipeline
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

# Train model
baseline_model.fit(X_train_base, y_train)

# Predict probabilities
baseline_predictions = baseline_model.predict_proba(X_test_base)[:, 1]

# Calculate ROC AUC
baseline_auc = roc_auc_score(y_test, baseline_predictions)

print("Our Baseline ROC AUC:", round(baseline_auc, 4))

Our Baseline ROC AUC: 0.7096


In [31]:
# Features available at booking creation
numeric_features = [
    "lead_time",
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled"
]

categorical_features = [
    "hotel",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "deposit_type",
    "customer_type",
    "agent",
    "company"
]

all_features = numeric_features + categorical_features

X_train_full = X_train[all_features].copy()
X_test_full = X_test[all_features].copy()

print("Number of features:", len(all_features))
print("Training shape:", X_train_full.shape)
print("Testing shape:", X_test_full.shape)

print("\nMissing values in training data:")
print(X_train_full.isnull().sum().sort_values(ascending=False).head(10))

Number of features: 23
Training shape: (95512, 23)
Testing shape: (23878, 23)

Missing values in training data:
company                      90111
agent                        13034
country                        391
children                         4
lead_time                        0
arrival_date_year                0
arrival_date_month               0
stays_in_week_nights             0
stays_in_weekend_nights          0
arrival_date_day_of_month        0
dtype: int64


In [32]:
numeric_features = [
    "lead_time",
    "arrival_date_year",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled"
]

categorical_features = [
    "hotel",
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "deposit_type",
    "customer_type",
    "agent",
    "company"
]

all_features = numeric_features + categorical_features

X_train_full = X_train[all_features].copy()
X_test_full = X_test[all_features].copy()

print("Number of features:", len(all_features))
print("Training shape:", X_train_full.shape)
print("Testing shape:", X_test_full.shape)

Number of features: 23
Training shape: (95512, 23)
Testing shape: (23878, 23)


In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score

# Numeric preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Create complete model
full_logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

print("Model pipeline created successfully!")

Model pipeline created successfully!


In [34]:
cv_scores = cross_val_score(
    full_logistic_model,
    X_train_full,
    y_train,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

print("Cross-validation scores:", cv_scores)
print("Mean CV ROC AUC:", round(cv_scores.mean(), 4))
print("Standard deviation:", round(cv_scores.std(), 4))

Cross-validation scores: [0.87773797 0.88179208 0.87823953 0.87794919 0.87571558]
Mean CV ROC AUC: 0.8783
Standard deviation: 0.002


In [35]:
# Train model
full_logistic_model.fit(X_train_full, y_train)

# Predict cancellation probabilities
full_predictions = full_logistic_model.predict_proba(
    X_test_full
)[:, 1]

# Calculate test ROC AUC
full_auc = roc_auc_score(y_test, full_predictions)

print("Baseline Test AUC:", round(baseline_auc, 4))
print("Improved Logistic Test AUC:", round(full_auc, 4))
print("Improvement:", round(full_auc - baseline_auc, 4))

Baseline Test AUC: 0.7096
Improved Logistic Test AUC: 0.8781
Improvement: 0.1684


In [36]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest pipeline
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=18,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest model created!")

Random Forest model created!


In [37]:
rf_cv_scores = cross_val_score(
    rf_model,
    X_train_full,
    y_train,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

print("Random Forest CV scores:", rf_cv_scores)
print("Mean Random Forest CV AUC:", round(rf_cv_scores.mean(), 4))
print("CV Standard Deviation:", round(rf_cv_scores.std(), 4))

Random Forest CV scores: [0.91316589 0.91708542 0.91302799 0.91294632 0.91053111]
Mean Random Forest CV AUC: 0.9134
CV Standard Deviation: 0.0021


In [38]:
# Train Random Forest
rf_model.fit(X_train_full, y_train)

# Predict probabilities
rf_predictions = rf_model.predict_proba(X_test_full)[:, 1]

# Calculate test AUC
rf_auc = roc_auc_score(y_test, rf_predictions)

print("Baseline AUC:", round(baseline_auc, 4))
print("Logistic Regression AUC:", round(full_auc, 4))
print("Random Forest AUC:", round(rf_auc, 4))

Baseline AUC: 0.7096
Logistic Regression AUC: 0.8781
Random Forest AUC: 0.9144


In [39]:
results = pd.DataFrame({
    "Model": [
        "Baseline Logistic Regression",
        "Full Logistic Regression",
        "Random Forest"
    ],
    "Test ROC AUC": [
        baseline_auc,
        full_auc,
        rf_auc
    ]
})

results = results.sort_values(
    by="Test ROC AUC",
    ascending=False
)

print(results)

                          Model  Test ROC AUC
2                 Random Forest      0.914354
1      Full Logistic Regression      0.878080
0  Baseline Logistic Regression      0.709633


## Experiment Log

| Experiment | Features | Model | Test ROC AUC |
|---|---|---|---|
| Baseline | 6 numeric features | Logistic Regression | 0.7096 |
| Experiment 1 | 23 booking-time features | Logistic Regression | 0.8781 |
| Experiment 2 | 23 booking-time features | Random Forest | 0.9144 |

### Cross-Validation Results

| Model | Mean CV ROC AUC | Standard Deviation |
|---|---:|---:|
| Full Logistic Regression | 0.8783 | 0.0020 |
| Random Forest | 0.9134 | 0.0021 |

### Best Model

Random Forest achieved the highest test ROC AUC of 0.9144.

Adding booking-time features significantly improved performance over the six-feature baseline. Random Forest further improved performance by capturing nonlinear relationships and interactions between features.

## Excluded Features and Reasons

The following features were excluded to prevent data leakage and ensure predictions use information available when the booking is created.

| Feature | Reason for exclusion |
|---|---|
| reservation_status | Directly reveals whether the reservation was cancelled or checked out. |
| reservation_status_date | Records information about the final reservation status. |
| assigned_room_type | May reflect room assignment decisions made after booking. |
| booking_changes | Can include modifications made after the original booking. |
| days_in_waiting_list | May not be known at the time of initial booking. |
| required_car_parking_spaces | May reflect requests made after booking. |
| total_of_special_requests | May be updated after the original booking. |

These features were excluded to simulate a realistic prediction scenario at booking creation.

## Final Conclusion

The original baseline Logistic Regression achieved a test ROC AUC of 0.7096.

After adding booking-time features and applying preprocessing techniques,
the Full Logistic Regression model achieved 0.8781.

The Random Forest model achieved the highest test ROC AUC of 0.9144,
with a mean 5-fold cross-validation ROC AUC of 0.9134.

The experiments demonstrate that feature engineering, categorical
encoding, missing-value handling, and nonlinear machine learning models
can substantially improve hotel booking cancellation prediction.

Final Best Model: Random Forest

Final Test ROC AUC: 0.9144

Baseline Test ROC AUC: 0.7096